# ECLIPSE · Notebook 02 — Model Training

End-to-end training run for ECLIPSE-PRIME on processed TCE tensors.
Verifies: model parameter count, forward pass shape, loss computation, one training epoch.

**Run preprocessing first:** `python -m src.preprocessing.dataset_builder`

In [ ]:
import sys; sys.path.insert(0, '..')
import torch
from src.models.eclipse_prime import ECLIPSEPrime
from src.utils.config import DEFAULT_CONFIG

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Build model and count parameters
model = ECLIPSEPrime.from_config(DEFAULT_CONFIG).to(device)
param_counts = model.parameter_count()
print('ECLIPSE-PRIME Parameter Counts:')
for k, v in param_counts.items():
    print(f'  {k:15s}: {v:>10,}')

In [ ]:
# Test forward pass with dummy inputs
B = 4  # batch size
T = DEFAULT_CONFIG.model.T_max

raw_flux    = torch.randn(B, T).to(device)
global_view = torch.randn(B, 2001).to(device)
local_view  = torch.randn(B, 201).to(device)
stellar     = torch.randn(B, 8).to(device)
centroid    = torch.randn(B, 201).to(device)

model.eval()
with torch.no_grad():
    out = model(raw_flux, global_view, local_view, stellar, centroid)

print('Forward pass outputs:')
for k, v in out.items():
    if hasattr(v, 'shape'):
        print(f'  {k:25s}: {tuple(v.shape)}')
    elif v is not None:
        print(f'  {k:25s}: {type(v).__name__}')

In [ ]:
# Test loss computation
from src.training.losses import ECLIPSEMultiTaskLoss
import torch

criterion = ECLIPSEMultiTaskLoss()

targets = {
    'class':      torch.randint(0, 4, (B,)).to(device),
    'period':     torch.rand(B).to(device) * 10,
    'duration':   torch.rand(B).to(device) * 0.2,
    'depth':      torch.rand(B).to(device) * 0.01,
    'snr':        torch.rand(B).to(device) * 20,
    'has_params': (torch.rand(B) > 0.5).to(device),
    'local_view': local_view,
}

model.train()
out = model(raw_flux, global_view, local_view, stellar, centroid)
losses = criterion(out, targets)
print('Loss components:')
for k, v in losses.items():
    print(f'  {k:12s}: {v.item():.4f}')

In [ ]:
# One backward pass (gradient check)
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=3e-4)
optimizer.zero_grad()
losses['total'].backward()

# Check gradients are flowing
grad_norms = {}
for name, p in model.named_parameters():
    if p.grad is not None:
        grad_norms[name] = p.grad.norm().item()

max_grad = max(grad_norms.values())
min_grad = min(grad_norms.values())
print(f'Gradient norms: min={min_grad:.2e}, max={max_grad:.2e}')
print(f'Number of params with gradients: {len(grad_norms)}')
print('✓ Backward pass successful — gradients flowing through all layers')

In [ ]:
# Launch full training (requires processed data)
import os
from pathlib import Path

train_csv = '../data/processed/tce_catalog_train.csv'
val_csv   = '../data/processed/tce_catalog_val.csv'

if Path(train_csv).exists() and Path(val_csv).exists():
    from src.training.data_loader import make_eclipse_dataloader
    from src.training.train import train_eclipse

    train_loader = make_eclipse_dataloader(train_csv, split='train', augment=True, batch_size=16)
    val_loader   = make_eclipse_dataloader(val_csv,   split='val',   augment=False, batch_size=16)
    print(f'Train: {len(train_loader.dataset)} samples, Val: {len(val_loader.dataset)} samples')

    best_f1 = train_eclipse(train_loader, val_loader)
    print(f'Training complete. Best val macro F1: {best_f1:.4f}')
else:
    print(f'Processed data not found at {train_csv}')
    print('Run the preprocessing pipeline first:')
    print('  python -c "from src.preprocessing.dataset_builder import build_tce_catalog; build_tce_catalog(sector=1)"')